# 03 — Démonstration de l'infrastructure de suivi (Neon + S3)

**Ce notebook ne produit aucun modèle.** Il n'entraîne rien, n'écrase rien, n'écrit rien
dans la base ni dans le bucket. Il **vérifie** que la chaîne de suivi d'expériences
fonctionne, et rend visible ce qui, sinon, resterait invisible : le fait que les runs
MLflow vivent sur une infrastructure **distante**, pas sur cette machine.

Il s'exécute en quelques secondes et peut être rejoué devant un examinateur.

---

## Pourquoi ce notebook existe

Une interface MLflow remplie de runs ressemble exactement à... une interface MLflow
remplie de runs. Rien ne distingue visuellement un file store local d'un backend
PostgreSQL distant. Ce notebook est la première des **trois vues** qui, mises en regard,
rendent le distant démontrable :

| Vue | Outil | Chemin technique |
|---|---|---|
| **1 — ce notebook** | Python | `mlflow` -> `psycopg2` -> Neon |
| 2 — console Neon | Navigateur | console de l'hébergeur -> Neon |
| 3 — `mlflow ui` | Serveur MLflow | SQLAlchemy -> Neon |

Aucun de ces trois chemins ne passe par les deux autres. S'ils affichent la même chose,
la seule explication est qu'ils lisent la même base distante.

---

## Rappel de l'architecture

| Donnée | Stockage | Variable |
|---|---|---|
| Métriques, paramètres, métadonnées | **PostgreSQL / Neon** | `MLFLOW_TRACKING_URI` |
| Modèle sérialisé + environnement | **AWS S3** | `MLFLOW_ARTIFACT_LOCATION` |

Neon conserve pour chaque run l'**URI S3** de ses artefacts (colonne `artifact_uri`) :
c'est ce qui relie les deux stockages.

---
## 0 · Configuration

Le `.env` est chargé ici comme dans `train.py` : `override=False`, donc une variable
déjà présente dans l'environnement n'est jamais écrasée par le fichier. C'est la
sémantique de production — en conteneur il n'y a pas de `.env`, l'appel ne fait rien,
et ce sont les variables injectées par la plateforme qui servent.

⚠️ **Sécurité** : l'URI de tracking contient le mot de passe Neon. La fonction `masque()`
ne conserve que la partie située **après** le `@` (hôte, base, paramètres). À utiliser
systématiquement avant tout affichage — les sorties de cellules sont sauvegardées dans
le fichier `.ipynb`, qui est versionné.

In [ ]:
import os
from pathlib import Path

import mlflow
import pandas as pd
from dotenv import load_dotenv

# Le notebook vit dans model/, à côté du .env — même dossier que train.py.
HERE = Path.cwd()
load_dotenv(HERE / ".env", override=False)

TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI")
ARTIFACT_LOCATION = os.environ.get("MLFLOW_ARTIFACT_LOCATION")

def masque(uri: str) -> str:
    """Ne conserve que le domaine et le nom de la base.

    Le split sur '@' retire déjà les identifiants, mais l'identifiant du
    endpoint Neon (ep-xxxx) reste : c'est une donnée d'infrastructure qui
    n'a pas sa place dans un dépôt public, les sorties de cellules étant
    enregistrées dans le .ipynb.
    """
    if not uri:
        return "NON DÉFINI (repli local)"
    hote, _, reste = uri.split("@")[-1].partition("/")
    base = reste.split("?")[0]
    return f"…{'.'.join(hote.split('.')[-3:])}/{base}"

print("Backend store (Neon) :", masque(TRACKING_URI))
print("Artifact store (S3)  :", ARTIFACT_LOCATION or "NON DÉFINI (repli local)")

# Aucune écriture n'aura lieu : on ne fait que lire.
mlflow.set_tracking_uri(TRACKING_URI)

---
## 1 · Les expériences et leur `artifact_location`

C'est ici que se joue le point le plus instructif du projet.

L'`artifact_location` est une **colonne de la table `experiments`**, écrite **une seule
fois à la création** de l'expérience et jamais recalculée. Définir la variable
`MLFLOW_ARTIFACT_LOCATION` après coup n'a **aucun effet** — et **sans message d'erreur**.
Ce qui compte n'est pas la variable, mais le fait qu'elle soit **transmise** à
`create_experiment(name, artifact_location=...)`, ce que fait `setup_experiment()`
dans `train.py`.

**La contre-épreuve est visible dans la sortie ci-dessous** : l'expérience `Default`,
créée en interne par le moteur MLflow **sans** ce paramètre, conserve un emplacement
`file:` **local**, alors que la variable S3 était pourtant définie. Mes expériences,
créées par mon code, pointent bien vers `s3://`.

Deux expériences, même base, même variable d'environnement, deux résultats.

In [ ]:
# search_experiments() interroge la table `experiments` de Neon.
exps = mlflow.search_experiments()

pd.DataFrame([
    {
        "id": e.experiment_id,
        "expérience": e.name,
        "artifact_location": e.artifact_location,
        "stockage": "S3 ✅" if e.artifact_location.startswith("s3://") else "LOCAL ⚠️",
    }
    for e in exps
]).sort_values("id")

---
## 2 · Le balayage vu en SQL — ce qu'un bucket ne permettrait pas

Requête directe sur Neon, sans passer par MLflow. Elle répond à la question
« pourquoi ne pas tout stocker dans S3 ? » : on ne peut pas interroger, croiser ni
dériver des colonnes sur un bucket.

Deux points de lecture :

- **`FILTER (WHERE ...)`** est une agrégation conditionnelle. Les tables `params` et
  `metrics` sont en format **long** (une ligne par couple clé/valeur) ; ce mécanisme
  les repivote en colonnes.
- **`ecart_surapprentissage` n'existe nulle part** : elle est *calculée* par la requête.
  C'est exactement ce qu'apporte un backend relationnel.

In [ ]:
REQUETE = """
SELECT r.name AS run,
       (MAX(p.value) FILTER (WHERE p.key = 'max_depth'))::int AS max_depth,
       ROUND((MAX(m.value) FILTER (WHERE m.key = 'train_rmse'))::numeric, 2) AS train_rmse,
       ROUND((MAX(m.value) FILTER (WHERE m.key = 'test_rmse'))::numeric, 2)  AS test_rmse,
       ROUND((MAX(m.value) FILTER (WHERE m.key = 'test_rmse')
            - MAX(m.value) FILTER (WHERE m.key = 'train_rmse'))::numeric, 2) AS ecart_surapprentissage,
       r.artifact_uri
FROM runs r
JOIN experiments e ON e.experiment_id = r.experiment_id
LEFT JOIN params  p ON p.run_uuid = r.run_uuid
LEFT JOIN metrics m ON m.run_uuid = r.run_uuid
WHERE e.name = 'getaround-pricing-sweep'
GROUP BY r.run_uuid, r.name, r.artifact_uri
ORDER BY 2;
"""

# create_engine lit l'URI postgresql:// et choisit psycopg2 comme driver.
from sqlalchemy import create_engine

engine = create_engine(TRACKING_URI)
sweep = pd.read_sql(REQUETE, engine)
sweep

**Ce que montre le tableau** : `train_rmse` décroît continûment avec la profondeur —
la forêt mémorise de mieux en mieux le jeu d'entraînement — tandis que `test_rmse`
descend, atteint un plancher, puis cesse de progresser. La colonne
`ecart_surapprentissage`, elle, ne cesse de croître.

Au-delà du plancher, on paie en **taille de modèle** (62 Mo sans plafond contre 2,3 Mo
à `max_depth=12`) sans rien gagner en généralisation. C'est l'arbitrage
performance / coût de déploiement qui justifie le réglage retenu — une décision
**mesurée**, pas affirmée.

Noter aussi la colonne `artifact_uri` : elle commence par `s3://` et **diffère pour
chaque run**. C'est le pont entre les deux stockages, matérialisé dans une colonne.

---
## 3 · Recharger un modèle depuis S3

La démonstration la plus directe que l'artifact store fonctionne : un modèle entraîné
il y a plusieurs jours, rechargé **depuis le cloud** — sans le fichier local — et qui
prédit.

Ce qui est stocké sur S3 n'est pas seulement `model.pkl` mais aussi `MLmodel`,
`conda.yaml`, `python_env.yaml` et `requirements.txt` : **l'environnement est enregistré
avec le modèle**, c'est ce qui le rend rechargeable ailleurs.

*(Note MLflow 3 : un modèle loggé est une entité de premier rang, rangée sous
`models/m-<hash>/` et identifiée séparément du run. La cellule résout l'emplacement
automatiquement plutôt que de coder un chemin en dur.)*

In [ ]:
import mlflow.sklearn

# On prend le run le moins profond du balayage (modèle le plus léger à télécharger).
exp = mlflow.get_experiment_by_name("getaround-pricing-sweep")
runs = mlflow.search_runs(experiment_ids=[exp.experiment_id],
                          order_by=["params.max_depth ASC"])
run_id = runs.iloc[0]["run_id"]
print("Run choisi :", runs.iloc[0]["tags.mlflow.runName"], "|", run_id)

# "runs:/<id>/model" est une URI logique : MLflow la résout vers l'emplacement
# physique réel (ici S3) en interrogeant le backend store. On ne code jamais
# le chemin S3 en dur -> c'est tout l'intérêt d'avoir un backend qui fait le lien.
pipe = mlflow.sklearn.load_model(f"runs:/{run_id}/model")
print("Modèle rechargé depuis S3 :", type(pipe).__name__)

In [ ]:
# Prédiction sur la voiture de référence — les MÊMES features brutes que celles
# envoyées à l'API. Le pipeline embarque le prétraitement : aucun encodage manuel.
voiture = pd.DataFrame([{
    "model_key": "Citroën", "mileage": 140000, "engine_power": 100,
    "fuel": "diesel", "paint_color": "black", "car_type": "sedan",
    "private_parking_available": True, "has_gps": True,
    "has_air_conditioning": False, "automatic_car": False,
    "has_getaround_connect": True, "has_speed_regulator": True,
    "winter_tires": False,
}])

print(f"Prix prédit : {pipe.predict(voiture)[0]:.2f} €/jour")

**À souligner** : ce modèle vient de S3, celui que sert l'API vient d'un fichier local
embarqué dans son image Docker. Ce sont **deux sérialisations indépendantes** du même
pipeline, produites par deux appels que rien ne relie (`mlflow.sklearn.log_model()` d'un
côté, `joblib.dump()` de l'autre).

Conséquence assumée : **l'API ne dépend ni de MLflow, ni de Neon, ni de S3**. Si toute
l'infrastructure de suivi tombait, elle continuerait de servir ses prédictions. C'est
précisément ce lien qu'un **Model Registry** créerait — au prix d'une dépendance réseau
au démarrage.

---
## 4 · La courbe que MLflow ne sait pas tracer

L'interface MLflow ne superpose pas deux métriques sur un même graphique en fonction
d'un hyperparamètre : il faudrait deux captures d'écran séparées.

Ici, on la construit depuis le DataFrame de l'étape 2 — c'est-à-dire **depuis les
données de suivi elles-mêmes**. Le graphique n'est pas une illustration importée : il
est *dérivé* de la base. Meilleure justification possible du choix d'un backend SQL.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4.5))

ax.plot(sweep["max_depth"], sweep["train_rmse"], marker="o", label="train_rmse")
ax.plot(sweep["max_depth"], sweep["test_rmse"], marker="o", label="test_rmse")

# La zone entre les deux courbes EST le surapprentissage : plus elle est large,
# plus le modèle mémorise le train sans mieux généraliser.
ax.fill_between(sweep["max_depth"], sweep["train_rmse"], sweep["test_rmse"],
                alpha=0.12, label="écart = surapprentissage")

# Repère sur la valeur retenue en production.
ax.axvline(12, linestyle="--", linewidth=1, color="grey")
ax.annotate("max_depth = 12\n(valeur retenue)", xy=(12, ax.get_ylim()[1]),
            xytext=(4, -14), textcoords="offset points", fontsize=9, va="top")

ax.set_xlabel("max_depth")
ax.set_ylabel("RMSE (€)")
ax.set_title("Balayage de la profondeur — RMSE train vs test")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

**Lecture** : la courbe `train_rmse` s'effondre avec la profondeur (mémorisation
croissante) tandis que `test_rmse` atteint un plancher. L'aire entre les deux — le
surapprentissage — ne cesse de s'élargir.

Comparaison utile avec la régression linéaire de l'expérience `getaround-pricing` :

| | RMSE train | RMSE test | écart |
|---|---|---|---|
| RandomForest (`depth=12`) | ≈ 9,3 | ≈ 17,0 | **+7,7** |
| Régression linéaire | ≈ 17,8 | ≈ 18,4 | **+0,55** |

La forêt sur-apprend, la linéaire pas du tout — elle est trop contrainte pour mémoriser
quoi que ce soit, c'est du **biais** pur. Ce qui tranche reste la performance **en test** :
le surapprentissage de la forêt est le prix acceptable d'une meilleure généralisation.

---

## Pour finir la démonstration

Ce notebook est la **vue 1**. Enchaîner ensuite sur :

- **vue 2** — la console Neon, où la requête de l'étape 2 renvoie les mêmes runs,
  depuis un navigateur qui ne touche à rien de cette machine ;
- **vue 3** — `mlflow ui --backend-store-uri $env:MLFLOW_TRACKING_URI`, qui affiche
  exactement les mêmes runs sur `localhost:5000`.

> *Trois outils qui n'ont rien en commun, une seule source. Le suivi ne vit pas sur
> ma machine — il vit dans la base.*